# FinTrust Week 1 — Data Profiling

AnalystLab Africa — Data Analytics

Data Understanding: The purpose of this notebook is to understand the given datasets.

In [3]:
import pandas as pd
import numpy as np

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
cust = pd.read_csv("/content/drive/MyDrive/ANALYSTLAB MATERIALS/WEEK 1/FinTrust_Customer_Data - FinTrust_Customer_Data.csv")
txn = pd.read_csv("/content/drive/MyDrive/ANALYSTLAB MATERIALS/WEEK 1/FinTrust_Transaction_Data - FinTrust_Transaction_Data.csv")

In [15]:
print(f"Customer data:    {cust.shape[0]} rows, {cust.shape[1]} columns")
print(f"Transaction data: {txn.shape[0]} rows, {txn.shape[1]} columns")

Customer data:    1500 rows, 12 columns
Transaction data: 12000 rows, 11 columns


In [16]:
print("Customer data:\n", cust.dtypes)
print("\nTransaction data:\n", txn.dtypes)

Customer data:
 Customer_ID                  object
Customer_Name                object
Age                           int64
Gender                       object
City                         object
Customer_Segment             object
Account_Type                 object
Tenure_Months                 int64
Digital_Engagement_Score    float64
Monthly_Income_Band          object
Preferred_Channel            object
Account_Status               object
dtype: object

Transaction data:
 Transaction_ID                object
Customer_ID                   object
Transaction_DateTime          object
Transaction_Type              object
Amount_NGN                   float64
Channel                       object
Device_Type                   object
Location                      object
International_Transaction     object
Transaction_Status            object
Risk_Review_Flag              object
dtype: object


In [17]:
print("Customer data:\n", cust.isnull().sum())
print("\nTransaction data:\n", txn.isnull().sum())

Customer data:
 Customer_ID                 0
Customer_Name               0
Age                         0
Gender                      0
City                        0
Customer_Segment            0
Account_Type                0
Tenure_Months               0
Digital_Engagement_Score    0
Monthly_Income_Band         0
Preferred_Channel           0
Account_Status              0
dtype: int64

Transaction data:
 Transaction_ID                0
Customer_ID                   0
Transaction_DateTime          0
Transaction_Type              0
Amount_NGN                    0
Channel                       0
Device_Type                  96
Location                     96
International_Transaction     0
Transaction_Status            0
Risk_Review_Flag              0
dtype: int64


In [18]:
print(f"Duplicate Customer_ID values:     {cust['Customer_ID'].duplicated().sum()}")
print(f"Fully duplicate customer rows:     {cust.duplicated().sum()}")
print(f"Duplicate Transaction_ID values:   {txn['Transaction_ID'].duplicated().sum()}")
print(f"Fully duplicate transaction rows:  {txn.duplicated().sum()}")

Duplicate Customer_ID values:     0
Fully duplicate customer rows:     0
Duplicate Transaction_ID values:   0
Fully duplicate transaction rows:  0


In [19]:
orphans = (~txn["Customer_ID"].isin(cust["Customer_ID"])).sum()
print(f"Transaction rows with no matching customer: {orphans}")

Transaction rows with no matching customer: 0


In [20]:
for col in cust.select_dtypes(include="object").columns:
    if col not in ("Customer_ID", "Customer_Name"):
        print(f"\nCustomer.{col}:\n{cust[col].value_counts()}")

for col in txn.select_dtypes(include="object").columns:
    if col not in ("Transaction_ID", "Customer_ID", "Transaction_DateTime"):
        print(f"\nTransaction.{col}:\n{txn[col].value_counts()}")


Customer.Gender:
Gender
Male                 727
Female               722
Prefer not to say     51
Name: count, dtype: int64

Customer.City:
City
Lagos            488
Abuja            223
Port Harcourt    172
Kano             159
Ibadan           148
Enugu            108
Kaduna           103
Benin City        99
Name: count, dtype: int64

Customer.Customer_Segment:
Customer_Segment
Everyday    711
Premium     295
Student     278
SME         216
Name: count, dtype: int64

Customer.Account_Type:
Account_Type
Savings    893
Current    378
Premium    229
Name: count, dtype: int64

Customer.Monthly_Income_Band:
Monthly_Income_Band
100k-249k     430
250k-499k     416
Below 100k    265
500k-999k     255
1m+           134
Name: count, dtype: int64

Customer.Preferred_Channel:
Preferred_Channel
Mobile App    924
Web           357
USSD          219
Name: count, dtype: int64

Customer.Account_Status:
Account_Status
Active        1367
Dormant        107
Restricted      26
Name: count, dtype: int6

In [21]:
print("Customer numeric fields:\n", cust.describe())
print("\nAmount_NGN:\n", txn["Amount_NGN"].describe())

Customer numeric fields:
                Age  Tenure_Months  Digital_Engagement_Score
count  1500.000000    1500.000000               1500.000000
mean     41.587333      49.858667                 67.954000
std      13.733127      27.910990                 17.555376
min      18.000000       1.000000                 16.700000
25%      29.750000      26.000000                 56.100000
50%      42.000000      51.000000                 68.000000
75%      53.000000      74.000000                 80.925000
max      65.000000      96.000000                100.000000

Amount_NGN:
 count     12000.000000
mean      46706.446238
std       86028.715234
min         100.170000
25%        2192.220000
50%       10305.935000
75%       48484.475000
max      693454.450000
Name: Amount_NGN, dtype: float64


In [22]:
txn["Transaction_DateTime"] = pd.to_datetime(txn["Transaction_DateTime"])
print(f"Earliest transaction: {txn['Transaction_DateTime'].min()}")
print(f"Latest transaction:   {txn['Transaction_DateTime'].max()}")

Earliest transaction: 2026-01-01 00:00:00
Latest transaction:   2026-03-31 23:59:00


In [23]:
print("By channel (%):\n", pd.crosstab(
    txn["Channel"], txn["Risk_Review_Flag"], normalize="index") * 100)

print("\nBy international flag (%):\n", pd.crosstab(
    txn["International_Transaction"], txn["Risk_Review_Flag"], normalize="index") * 100)

By channel (%):
 Risk_Review_Flag         No        Yes
Channel                               
ATM               78.820836  21.179164
Mobile App        80.595845  19.404155
POS               81.654827  18.345173
USSD              82.677165  17.322835
Web               78.651685  21.348315

By international flag (%):
 Risk_Review_Flag                  No        Yes
International_Transaction                      
No                         81.119792  18.880208
Yes                        63.125000  36.875000


In [24]:
avg_txn_per_cust = txn.groupby("Customer_ID").size().mean()
print(f"Average transactions per customer: {avg_txn_per_cust:.2f}")

Average transactions per customer: 8.00
